# Assignment 3: Approximation via Monte Carlo

This is the **GenJAX (canonical)** stencil. Two paired stencils are available — `mc_approx_python.ipynb` (Python + numpy/scipy) and `mcmc_approx.Rmd` (R). Matlab available on request.

**How GenJAX is used here.** The sampler is a loop *you* assemble. GenJAX gives you the scoring primitives — `beta.sample`, `beta.logpdf`, `normal.logpdf` — and JAX gives you speed (`jax.vmap`, `jax.lax.scan`, `jax.jit`). This mirrors the lecture: MH is *assembled from the scoring primitives*, not a black box.

In this assignment you will explore three Monte Carlo methods:

1. **Problem 1** — compare **naive Monte Carlo** and **importance sampling** for estimating a tail probability $P(Y>2)$, $Y\sim N(0,1)$.
2. **Problem 2** — build a **Markov chain Monte Carlo** sampler (interleaving **Gibbs** and **Metropolis–Hastings**) for the hierarchical Beta-Binomial model of Kemp, Perfors & Tenenbaum (2007).
3. **Problem 3** — quantify how "efficient" a set of samples is with the **effective sample size** ($N_{\mathrm{eff}}$).

**Read `mc_approx.pdf` first** — it has the full problem statements and all the math. This notebook is the scaffold: cells marked `# fill me` are for you to complete. Submit the completed notebook (it must run end-to-end) *or* a single PDF report with your code, figures, and written answers.

**Textbook background:** Tutorial 3 Ch 12 (hierarchical Bayes / approximate inference). The Week 7 lecture covers Monte Carlo, importance sampling, and MCMC (Metropolis–Hastings + Gibbs).

In [ ]:
# !pip install genjax
import jax
import jax.numpy as jnp
import jax.random as random
from genjax import beta, normal
import matplotlib.pyplot as plt

# genjax 0.10.3 primitive cheatsheet (all match scipy):
#   beta.sample(key, a, b)        -> one Beta(a,b) draw
#   beta.logpdf(x, a, b)          -> log density (vectorizes over x)
#   normal.logpdf(x, mu, sigma)   -> log density of N(mu, sigma^2)
#   random.normal(key, shape)     -> standard normal draws
# NOTE: genjax's `binomial` takes LOGITS, not a probability -- we do not
# need it here (bag data are given directly as (y, n)).

---

## Problem 1: Monte Carlo vs. importance sampling

Estimate $P(Y>2)$ for $Y\sim N(0,1)$ two ways, with $T=1000$ samples.

- **(a) Naive MC.** Draw $x^{(1)},\dots,x^{(T)}\sim N(0,1)$ and form the cumulative estimate $f_{MC}(t) = \frac{1}{t}\sum_{s\le t} \mathbb{1}[x^{(s)}>2]$. Plot $f_{MC}$ vs. $t$. Report $f_{MC}(T)$ and explain the shape of the curve.
- **(b) Importance sampling.** Draw $x^{(1)},\dots,x^{(T)}\sim q=N(2,1)$ and form the cumulative IS estimate using the explicit weight $e^{-2x+2}\,\mathbb{1}[x>2]$. Plot $f_{IS}$ vs. $t$. Report $f_{IS}(T)$.
- **(c)** Compare the two curves: how and why do they differ? State a broad lesson about Monte Carlo approximation.

In [ ]:
# fill me -- Problem 1(a) and 1(b)
#
# (a) Naive MC for P(Y>2), Y~N(0,1):
#   - draw x ~ N(0,1), T of them, with random.normal
#   - indicator = (x > 2)
#   - cumulative estimate f_mc[t] = mean of indicator over first t samples
#     (hint: jnp.cumsum(indicator) / jnp.arange(1, T+1))
#
# (b) Importance sampling with q = N(2,1):
#   - draw xis = 2 + random.normal(...)
#   - weight w = exp(-2*xis + 2) * (xis > 2)   # the explicit p/q * indicator
#   - cumulative estimate f_is[t] = mean of w over first t samples

T = 1000
key = random.PRNGKey(0)

# f_mc = ...
# f_is = ...

# Plot both cumulative estimates vs t on the same axes, with a line at the
# true value (you may use jax.scipy.stats.norm or scipy.stats.norm for the truth).


---

## Problem 2: MCMC for the Kemp hierarchical Beta-Binomial model

You observe $M$ bags of marbles; bag $i$ shows $y_i$ white out of $n_i$ drawn. The model:
$$\theta_i \mid \kappa,\varphi \sim \mathrm{Beta}(\kappa\varphi,\ \kappa(1-\varphi)), \qquad y_i \mid n_i,\theta_i \sim \mathrm{Binomial}(\theta_i; n_i),$$
parameterized by the **mean** $\varphi=a/(a+b)$ and **concentration** $\kappa=a+b$ (so $a=\kappa\varphi$, $b=\kappa(1-\varphi)$). Priors: $\varphi\sim\mathrm{Uniform}(0,1)$ and a weak proper **log-normal** on $\kappa$, i.e. $\log\kappa\sim N(\mu_0,\sigma_0^2)$ with $\mu_0=\log 10$, $\sigma_0=1.5$.

The sampler does two moves per sweep:
- **Gibbs** resamples each $\theta_i$ from its conjugate posterior $\mathrm{Beta}(\kappa\varphi+y_i,\ \kappa(1-\varphi)+n_i-y_i)$.
- **Metropolis–Hastings** updates $(\varphi,\ell)$ with $\ell=\log\kappa$, via a symmetric Gaussian random walk. The acceptance ratio is the Beta likelihood of the current $\theta_i$'s times the log-normal prior ratio on $\ell$ (the proposal is symmetric, so there is no asymmetry-correction term, and there is no Jacobian — see the PDF).

See `mc_approx.pdf` Problem 2 for the full derivation and the acceptance formula.

In [ ]:
# Provided: the two bag sets and the prior hyperparameters.
MU0, S0 = jnp.log(10.0), 1.5   # log-normal prior on kappa: log(kappa) ~ N(MU0, S0^2)

def ab(kappa, phi):
    """Convert (concentration, mean) -> Beta shape params (a, b)."""
    return kappa * phi, kappa * (1 - phi)

# Bag set I: 10 bags, each 9 white of 20.   Bag set II: 5 bags 1/20, 5 bags 19/20.
bagset_I  = (jnp.array([9.0]*10),               jnp.array([20.0]*10))
bagset_II = (jnp.array([1.0]*5 + [19.0]*5),     jnp.array([20.0]*10))

In [ ]:
# fill me -- Problem 2(a): the conjugate posterior over theta for FIXED (kappa, phi).
#
# For each (kappa, phi) pair in the PDF, plot on one figure: the prior Beta(a, b),
# and the posteriors after (1 white, 0 black), (5,5), and (9,1). Use the conjugate
# update: posterior = Beta(a + y, b + (n - y)). beta.logpdf(grid, a, b) gives the
# log-density on a grid in (0,1); exponentiate to plot.

grid = jnp.linspace(1e-3, 1 - 1e-3, 400)

# def plot_posterior_updates(kappa, phi): ...
# plot_posterior_updates(1.0, 0.5)   # etc. for the six pairs


In [ ]:
# fill me -- Problem 2(b)+(c): implement the Gibbs + MH sampler.
#
# Write ONE sweep as a function step(carry, key) for use with jax.lax.scan, where
# carry = (phi, ell) with ell = log(kappa). Each sweep:
#
#   1. GIBBS: resample every theta_i from its conjugate posterior
#        theta_i ~ Beta(a + y_i,  b + (n_i - y_i))     where a, b = ab(exp(ell), phi)
#      Vectorize over bags: split a key per bag and jax.vmap beta.sample.
#
#   2. MH PROPOSE: eps ~ N(0,1) of shape (2,);
#        phi_p = phi + s_phi * eps[0];   ell_p = ell + s_ell * eps[1]
#      (s_phi, s_ell are the step sizes you will tune in part (e).)
#
#   3. ACCEPT/REJECT: with ap, bp = ab(exp(ell_p), phi_p),
#        log_c =   sum(beta.logpdf(theta, ap, bp)) - sum(beta.logpdf(theta, a, b))
#                + normal.logpdf(ell_p, MU0, S0)    - normal.logpdf(ell, MU0, S0)
#      accept iff (0 < phi_p < 1) and (log(uniform) < log_c). On reject, keep (phi, ell).
#      (Symmetric proposal => no asymmetry correction; prior is on ell directly => no Jacobian.)
#
#   Record (kappa, phi, accepted) each sweep via the scan output.
#
# Then wrap with jax.lax.scan over random.split(key, T). Remember to jit with
# T as a STATIC argument (functools.partial(jax.jit, static_argnums=...)) -- otherwise
# random.split(key, T) fails to trace.

def make_sampler(y, n, s_phi, s_ell):
    M = len(y)
    def step(carry, key):
        phi, ell = carry
        # fill me
        pass
    def run(key, T):
        # fill me -- lax.scan(step, (0.5, jnp.log(10.0)), random.split(key, T))
        pass
    return run


In [ ]:
# fill me -- Problem 2(d): run the sampler on bag sets I and II.
#
# For each set: run T = 3000 sweeps (after a few hundred burn-in), then
#   - histogram the kappa samples and the phi samples (separately),
#   - report posterior means kbar = mean(kappa), pbar = mean(phi),
#   - report the predictive prob a new bag is white = pbar (since E[theta]=phi).
# Suggested starting step sizes: set I  ~ (s_phi=0.04, s_ell=0.30);
#                                set II ~ (s_phi=0.05, s_ell=0.40).

# run_I = make_sampler(*bagset_I, s_phi=0.04, s_ell=0.30)
# ks, ps, accs = run_I(random.PRNGKey(1), 3000)
# ... burn-in, histograms, means ...


In [ ]:
# fill me -- Problem 2(e): tune the step sizes.
#
# Report the acceptance rate (mean of the `accepted` flags) for each bag set, and
# adjust (s_phi, s_ell) so each lands in ~0.2-0.5. You will find the two sets prefer
# different step sizes. Then write the short explanation asked for in the PDF
# (too-small vs too-large step; why the two sets differ; connect to mixing).


---

## Problem 3: Effective sample size

The effective sample size summarizes how useful a set of weighted samples is:
$$N_{\mathrm{eff}} = \frac{1}{\sum_t (w^{(t)})^2}, \qquad w^{(t)}\ge 0,\ \textstyle\sum_t w^{(t)}=1.$$

- **(a) Good vs. bad proposal.** With target $p=N(0,1)$ and IS weights $w\propto p/q$, compute $N_{\mathrm{eff}}$ for a **good** proposal that overlaps $p$ well (e.g. $q=N(0,1.5^2)$) and a **bad** one that does not (e.g. $q=N(4,1)$). You should see good $q$ in the high hundreds, bad $q$ in the single digits. (Do **not** use $q=N(2,1)$ here — that is part (b).)
- **(b) The puzzle.** Compute $N_{\mathrm{eff}}$ for two estimators of $P(Y>2)$: the IS sampler with $q=N(2,1)$ ($w\propto p/q$), and plain MC from $p$ ($w=1/T$). MC reports $N_{\mathrm{eff}}=T=1000$ while IS reports ~50 — yet IS is the *more accurate* estimator (Problem 1). Resolve the puzzle in writing (see the PDF for the three points to address).

In [ ]:
# fill me -- Problem 3.
#
def ess(w):
    """Effective sample size from unnormalized nonnegative weights w."""
    # fill me:  normalize w to sum to 1, then return 1 / sum(w**2)
    pass

# (a) good vs bad proposal for target p = N(0,1), weights w = p(x)/q(x):
#     good q = N(0, 1.5^2);  bad q = N(4, 1).  Use T = 1000. Report both ESS.
#     (weight in log space then exponentiate:
#         logw = normal.logpdf(xq, 0., 1.) - normal.logpdf(xq, q_mean, q_sd))
#
# (b) the puzzle: ESS for IS (q=N(2,1), w = p/q) vs plain MC (w = ones).
#     Report both, then write the resolution (see PDF).
